# Credit Risk Model Deployment — Phase 2: MLflow Experiment Tracking

**What MLflow does:** Logs every model run — parameters, metrics, artifacts — into a central tracking server with a visual UI. Enables side-by-side comparison of Model A vs Model B and registers the winning model as the official production version.

---
## Cell 1 — Install and import MLflow

In [1]:
# Install MLflow (run once)
import subprocess
subprocess.run(['pip', 'install', 'mlflow'], check=True)

import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix
)
from pathlib import Path

print(f'MLflow version: {mlflow.__version__}')
print('All imports successful.')

MLflow version: 3.12.0
All imports successful.


In [11]:
import os
for root, dirs, files in os.walk('..'):
    for f in files:
        if f == 'application_train.csv':
            print(os.path.join(root, f))

..\Credit-Risk-A-B-Test-Logistic-Regression-vs-LightGBM\Data\application_train.csv
..\Data\application_train.csv


---
## Cell 2 — Load models and test data

In [3]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print('Regenerating test set and predictions from source data...')

# ── Step 1: Load original dataset ────────────────────────────────────────────
# Update this path to match where your application_train.csv actually is
DATA_PATH = '../Data/application_train.csv'   # adjust if needed

df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]:,} rows')

# ── Step 2: Select features (same as Section 1) ───────────────────────────────
FEATURES = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'CNT_CHILDREN', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'REGION_RATING_CLIENT', 'TARGET'
]
df_model = df[FEATURES].copy()

# ── Step 3: Feature engineering (same as Section 1) ──────────────────────────
df_model['DEBT_TO_INCOME']    = df_model['AMT_CREDIT']   / (df_model['AMT_INCOME_TOTAL'] + 1)
df_model['PAYMENT_TO_INCOME'] = df_model['AMT_ANNUITY']  / (df_model['AMT_INCOME_TOTAL'] + 1)
df_model['LOAN_TO_VALUE']     = df_model['AMT_CREDIT']   / (df_model['AMT_GOODS_PRICE']  + 1)
df_model['CREDIT_TERM']       = df_model['AMT_CREDIT']   / (df_model['AMT_ANNUITY']      + 1)
df_model['AGE_YEARS']         = (-df_model['DAYS_BIRTH']) / 365
emp = df_model['DAYS_EMPLOYED'].replace(365243, 0)
df_model['EMPLOYED_YEARS']    = (-emp) / 365
df_model['EMPLOYED_YEARS']    = df_model['EMPLOYED_YEARS'].clip(lower=0)
df_model['EMPLOYMENT_TO_AGE'] = df_model['EMPLOYED_YEARS'] / (df_model['AGE_YEARS'] + 1)
df_model['EXT_SOURCE_MEAN']   = df_model[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)
df_model['EXT_SOURCE_MIN']    = df_model[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].min(axis=1)

# ── Step 4: Clean data ────────────────────────────────────────────────────────
cat_cols = df_model.select_dtypes(include='object').columns
num_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'TARGET']

for col in num_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# ── Step 5: Recreate exact same train/test split (seed=42 matches Section 1) ──
DROP_COLS   = ['TARGET', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
feature_cols = [c for c in df_model.columns if c not in DROP_COLS]

X = df_model[feature_cols]
y = df_model['TARGET']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Test set: {X_test.shape[0]:,} applicants, {X_test.shape[1]} features')

# ── Step 6: Load models and generate predictions ──────────────────────────────
model_a = joblib.load('models/model_a_logistic.pkl')
model_b = joblib.load('models/model_b_lightgbm.pkl')
scaler  = joblib.load('models/scaler.pkl')

X_test_scaled = scaler.transform(X_test)

y_prob_a = model_a.predict_proba(X_test_scaled)[:, 1]
y_prob_b = model_b.predict_proba(X_test)[:, 1]

# ── Step 7: Save for downstream use ──────────────────────────────────────────
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

predictions = pd.DataFrame({
    'y_true':   y_test.values,
    'y_prob_a': y_prob_a,
    'y_prob_b': y_prob_b
})
predictions.to_csv('predictions.csv', index=False)

with open('results/feature_cols.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print('X_test.csv, y_test.csv, predictions.csv saved.')
print(f'AUC Model A: {__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y_test, y_prob_a):.4f}')
print(f'AUC Model B: {__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y_test, y_prob_b):.4f}')

Regenerating test set and predictions from source data...
Dataset loaded: 307,511 rows
Test set: 92,254 applicants, 27 features
X_test.csv, y_test.csv, predictions.csv saved.
AUC Model A: 0.6992
AUC Model B: 0.7378


---
## Cell 3 — Configure MLflow tracking

MLflow stores all run data in a local folder called `mlruns/`. The tracking URI tells MLflow where to save. The experiment name groups related runs together — think of it as a project folder inside MLflow.

In [4]:
# Set tracking location — creates an mlruns/ folder in your project directory
mlflow.set_tracking_uri('file:./mlruns')

# Create or connect to the experiment
EXPERIMENT_NAME = 'credit_risk_ab_test'
mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Experiment ID: {experiment.experiment_id}')
print(f'Artifact location: {experiment.artifact_location}')

c:\Users\hanto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/12 18:38:23 INFO mlflow.tracking.fluent: Experiment with name 'credit_risk_ab_test' does not exist. Creating a new experiment.


Experiment: credit_risk_ab_test
Experiment ID: 197410562791891706
Artifact location: file:c:/Users/hanto/OneDrive/Desktop/Siqi Chen Projects/ML/credit_risk_ab_test/Credit-Risk-A-B-Test-Logistic-Regression-vs-LightGBM/mlruns/197410562791891706


---
## Cell 4 — Helper: compute all metrics

A single function that computes every metric we want to log for any model. Using the same function for both models ensures the logged metrics are strictly comparable.

In [5]:
def compute_metrics(y_true, y_prob, threshold, cost_fn=10000, cost_fp=500):
    """Compute the full metric set used across the A/B test project."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc         = roc_auc_score(y_true, y_prob)
    gini        = 2 * auc - 1
    ks          = float(np.max(tpr - fpr))
    total_cost  = int(fn * cost_fn + fp * cost_fp)

    return {
        'auc':               round(auc, 4),
        'gini':              round(gini, 4),
        'ks_statistic':      round(ks, 4),
        'avg_precision':     round(average_precision_score(y_true, y_prob), 4),
        'f1':                round(f1_score(y_true, y_pred), 4),
        'precision':         round(precision_score(y_true, y_pred), 4),
        'recall':            round(recall_score(y_true, y_pred), 4),
        'true_positives':    int(tp),
        'false_positives':   int(fp),
        'false_negatives':   int(fn),
        'true_negatives':    int(tn),
        'total_cost_usd':    total_cost,
        'cost_per_loan_usd': round(total_cost / len(y_true), 2),
    }

print('compute_metrics() ready.')

compute_metrics() ready.


---
## Cell 5 — Log Model A (Logistic Regression)

Every `with mlflow.start_run()` block creates one run entry in the MLflow UI. Inside the block we log: parameters (model settings), metrics (AUC, cost, etc.), and the model artifact (the `.pkl` file itself).

In [6]:
THRESHOLD_A = 0.261   # optimal threshold from Section 3

metrics_a = compute_metrics(y_test.values, y_prob_a, THRESHOLD_A)

with mlflow.start_run(run_name='model_a_logistic_regression') as run_a:

    # --- Parameters: what settings was this model trained with? ---
    mlflow.log_params({
        'model_type':        'LogisticRegression',
        'role':              'control',
        'C':                 0.1,
        'max_iter':          1000,
        'class_weight':      'balanced',
        'scaler':            'StandardScaler',
        'smote_strategy':    0.3,
        'train_test_split':  0.3,
        'optimal_threshold': THRESHOLD_A,
        'features_count':    len(feature_cols),
        'dataset':           'Home Credit Default Risk',
        'n_train_samples':   215258,
        'n_test_samples':    len(y_test),
    })

    # --- Metrics: how did it perform? ---
    mlflow.log_metrics(metrics_a)

    # --- Model artifact: save the actual model file ---
    signature = infer_signature(X_test_scaled, y_prob_a)
    mlflow.sklearn.log_model(
        sk_model   = model_a,
        artifact_path = 'model',
        signature  = signature,
        input_example = X_test_scaled[:3],
        registered_model_name = 'credit_risk_logistic_regression'
    )

    # --- Tags: metadata for filtering in the UI ---
    mlflow.set_tags({
        'model_family':  'linear',
        'section':       'section_2',
        'deployment_candidate': 'false',
        'regulatory_role': 'benchmark',
    })

    run_id_a = run_a.info.run_id

print(f'Model A logged — Run ID: {run_id_a}')
print(f'AUC: {metrics_a["auc"]}  |  Gini: {metrics_a["gini"]}  |  Cost: ${metrics_a["total_cost_usd"]:,}')

2026/05/12 18:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:38:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model A logged — Run ID: d9936a4e333441db9308b14f654b6e15
AUC: 0.6992  |  Gini: 0.3985  |  Cost: $37,665,000


c:\Users\hanto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'credit_risk_logistic_regression'.
Created version '1' of model 'credit_risk_logistic_regression'.


---
## Cell 6 — Log Model B (LightGBM)

Same structure as Model A — identical metric computation, same parameter logging pattern. The symmetry is deliberate: MLflow's comparison view works best when both runs log the same keys.

In [7]:
THRESHOLD_B = 0.168   # optimal threshold from Section 3

metrics_b = compute_metrics(y_test.values, y_prob_b, THRESHOLD_B)

with mlflow.start_run(run_name='model_b_lightgbm') as run_b:

    mlflow.log_params({
        'model_type':        'LightGBM',
        'role':              'treatment',
        'n_estimators':      500,
        'learning_rate':     0.05,
        'max_depth':         6,
        'num_leaves':        31,
        'min_child_samples': 50,
        'subsample':         0.8,
        'colsample_bytree':  0.8,
        'class_weight':      'balanced',
        'early_stopping':    50,
        'smote_strategy':    0.3,
        'train_test_split':  0.3,
        'optimal_threshold': THRESHOLD_B,
        'features_count':    len(feature_cols),
        'dataset':           'Home Credit Default Risk',
        'n_train_samples':   215258,
        'n_test_samples':    len(y_test),
    })

    mlflow.log_metrics(metrics_b)

    # Log statistical test results as metrics too
    mlflow.log_metrics({
        'delong_z_statistic': 18.40,
        'cohens_d':           12.85,
        'auc_lift_vs_baseline': round(metrics_b['auc'] - metrics_a['auc'], 4),
        'cost_saving_vs_baseline': metrics_a['total_cost_usd'] - metrics_b['total_cost_usd'],
    })

    signature = infer_signature(X_test, y_prob_b)
    mlflow.lightgbm.log_model(
        lgb_model     = model_b,
        artifact_path = 'model',
        signature     = signature,
        input_example = X_test.iloc[:3],
        registered_model_name = 'credit_risk_lightgbm'
    )

    mlflow.set_tags({
        'model_family':          'gradient_boosting',
        'section':               'section_2',
        'deployment_candidate':  'true',
        'explainability':        'shap',
        'regulatory_role':       'production',
    })

    run_id_b = run_b.info.run_id

print(f'Model B logged — Run ID: {run_id_b}')
print(f'AUC: {metrics_b["auc"]}  |  Gini: {metrics_b["gini"]}  |  Cost: ${metrics_b["total_cost_usd"]:,}')
print()
print(f'AUC lift:     +{metrics_b["auc"] - metrics_a["auc"]:.4f}')
print(f'Cost saving:  ${metrics_a["total_cost_usd"] - metrics_b["total_cost_usd"]:,}')

c:\Users\hanto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/12 18:39:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:39:07 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires 

Model B logged — Run ID: 05094ed3a12a4ea396a3d87b3279cc75
AUC: 0.7378  |  Gini: 0.4757  |  Cost: $34,194,500

AUC lift:     +0.0386
Cost saving:  $3,470,500


Successfully registered model 'credit_risk_lightgbm'.
Created version '1' of model 'credit_risk_lightgbm'.


---
## Cell 7 — Log the A/B test results as a separate run

The A/B test framework (DeLong test, bootstrap CI, threshold optimization) is itself a logged run — capturing the statistical validation layer separately from the individual model runs.

In [8]:
with mlflow.start_run(run_name='ab_test_statistical_validation') as run_ab:

    mlflow.log_params({
        'test_method':          'DeLong (1988)',
        'significance_level':   0.05,
        'bootstrap_iterations': 1000,
        'cost_fn_usd':          10000,
        'cost_fp_usd':          500,
        'threshold_candidates': 200,
        'control_model':        'LogisticRegression',
        'treatment_model':      'LightGBM',
    })

    mlflow.log_metrics({
        'delong_z_statistic':        18.40,
        'delong_p_value_lt':         0.001,
        'cohens_d':                  12.85,
        'auc_control':               metrics_a['auc'],
        'auc_treatment':             metrics_b['auc'],
        'auc_difference':            round(metrics_b['auc'] - metrics_a['auc'], 4),
        'optimal_threshold_control':  0.261,
        'optimal_threshold_treatment': 0.168,
        'cost_control_usd':          metrics_a['total_cost_usd'],
        'cost_treatment_usd':        metrics_b['total_cost_usd'],
        'cost_saving_usd':           metrics_a['total_cost_usd'] - metrics_b['total_cost_usd'],
        'projected_annual_saving_usd': 11_700_000,
        'h0_rejected':               1,    # 1 = True
        'significant_at_005':        1,
    })

    mlflow.set_tags({
        'run_type':      'statistical_validation',
        'verdict':       'lightgbm_wins',
        'recommendation': 'deploy_lightgbm_at_threshold_0.168',
    })

    # Save section3_results.csv as an artifact
    mlflow.log_artifact('results/section3_results.csv', artifact_path='ab_test_results')

    run_id_ab = run_ab.info.run_id

print(f'A/B test validation logged — Run ID: {run_id_ab}')

A/B test validation logged — Run ID: f3660fbf964040fc83108065c191fc08


---
## Cell 8 — Register production model with a stage

Model Registry is MLflow's version control for models. It gives LightGBM an official version number and a stage label (Staging → Production). This is the production-grade pattern used at companies like Databricks, Airbnb, and LinkedIn — a model is not 'in production' just because it's deployed, it must be registered and promoted through stages.

In [9]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the latest version of the LightGBM model
model_name    = 'credit_risk_lightgbm'
latest        = client.get_latest_versions(model_name)
latest_version = latest[0].version

# Add a description to the registered model
client.update_registered_model(
    name        = model_name,
    description = (
        'LightGBM credit default prediction model. '
        'AUC 0.7378 | Gini 0.4757 | KS 0.3547. '
        'Optimal threshold: 0.168. '
        'Validated via DeLong test (Z=18.40, p<0.001) and '
        '1000-iteration bootstrap CI. '
        'Projected annual cost saving vs baseline: $11.7M.'
    )
)

# Add version-specific notes
client.update_model_version(
    name        = model_name,
    version     = latest_version,
    description = (
        f'Version {latest_version} — trained on Home Credit dataset (307,511 applicants). '
        'SMOTE sampling_strategy=0.3. 500 estimators, lr=0.05, max_depth=6. '
        'Early stopping at 50 rounds. Registered from A/B test project.'
    )
)

print(f'Model registered: {model_name} v{latest_version}')
print()
print('Model Registry summary:')
print(f'  Name:    {model_name}')
print(f'  Version: {latest_version}')
print(f'  Run ID:  {run_id_b}')

Model registered: credit_risk_lightgbm v1

Model Registry summary:
  Name:    credit_risk_lightgbm
  Version: 1
  Run ID:  05094ed3a12a4ea396a3d87b3279cc75


C:\Users\hanto\AppData\Local\Temp\ipykernel_21816\2030870324.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest        = client.get_latest_versions(model_name)


---
## Cell 9 — Launch the MLflow UI

This opens the MLflow tracking server in your browser. You will see all three runs, their metrics, and the Model Registry.

In [10]:
print('='*55)
print('  MLflow UI instructions')
print('='*55)
print()
print('1. Open a NEW terminal (keep this notebook running)')
print('2. Navigate to your project folder')
print('3. Run this command:')
print()
print('   python -m mlflow ui --port 5000')
print()
print('4. Open your browser and go to:')
print()
print('   http://127.0.0.1:5000')
print()
print('What you will see:')
print('  - Experiments tab: all 3 runs with metrics side by side')
print('  - Compare button: select both model runs → visual comparison')
print('  - Models tab: registered credit_risk_lightgbm with version')
print('  - Artifacts: model files, feature importance, test results')
print()
print('='*55)
print('  Phase 2 complete!')
print('='*55)

  MLflow UI instructions

1. Open a NEW terminal (keep this notebook running)
2. Navigate to your project folder
3. Run this command:

   python -m mlflow ui --port 5000

4. Open your browser and go to:

   http://127.0.0.1:5000

What you will see:
  - Experiments tab: all 3 runs with metrics side by side
  - Compare button: select both model runs → visual comparison
  - Models tab: registered credit_risk_lightgbm with version
  - Artifacts: model files, feature importance, test results

  Phase 2 complete!


---
## Phase 2 Summary

| Step | What was logged |
|---|---|
| Model A run | Parameters, 13 metrics, model artifact, tags |
| Model B run | Parameters, 17 metrics (incl. DeLong + cost saving), model artifact, tags |
| A/B test run | Statistical test results, threshold optimization outputs, section3_results.csv |
| Model Registry | LightGBM registered as `credit_risk_lightgbm` with description and version notes |

**Interview talking point:** MLflow provides full experiment reproducibility — any run can be reloaded by its Run ID to reproduce the exact model, parameters, and metrics that led to the deployment decision. This is the audit trail that model risk teams require under SR 11-7.

**Next: Phase 3 — Docker containerization**